
# Thí nghiệm 02 — Chuẩn bị dữ liệu ERA5 Wind Speed

## 1. Khảo sát dữ liệu ERA5

### 1.1. Mục tiêu

Notebook này chuẩn bị dữ liệu tốc độ gió gần bề mặt từ bộ dữ liệu ERA5 tại 10 địa điểm thuộc nước Đức.

Quy trình bao gồm:

1. Đọc dữ liệu ERA5 từ tệp NetCDF.
2. Xác định hai thành phần vận tốc gió `u10` và `v10`.
3. Tính độ lớn tốc độ gió tại độ cao 10 m.
4. Lựa chọn 400 thời điểm đầu tiên.
5. Trích xuất chuỗi thời gian tại 10 địa điểm.
6. Kiểm tra dữ liệu và lưu dưới định dạng PyTorch.

Dữ liệu đầu ra được sử dụng trong `07_climate_era5_training.ipynb`.

### 1.2. Nguồn dữ liệu

Tệp được sử dụng:

`era5_germany_jan2020_10m_wind.nc`

ERA5 là bộ dữ liệu tái phân tích khí tượng, kết hợp thông tin quan trắc với mô hình dự báo số để xây dựng các trường dữ liệu khí quyển.

Khác với dữ liệu CMIP6/CORDEX trong những notebook trước, dữ liệu ở đây không thuộc kịch bản mô phỏng khí hậu RCP2.6.

### 1.3. Cấu trúc dữ liệu

Theo output đã lưu, tệp NetCDF chứa:

| Thuộc tính | Giá trị |
|---|---|
| Khoảng thời gian nguồn | 01/01/2020 – 31/01/2020 |
| Tần suất | 1 giờ |
| Số thời điểm | 744 |
| Số điểm vĩ độ | 33 |
| Số điểm kinh độ | 41 |
| Biến vận tốc gió | `u10`, `v10` |
| Đơn vị | m/s |

Hai biến có kích thước:

$$
U,V\in\mathbb{R}^{744\times33\times41}
$$

Trong đó:

- `u10`: thành phần vận tốc gió theo hướng Đông – Tây ở độ cao 10 m.
- `v10`: thành phần vận tốc gió theo hướng Bắc – Nam ở độ cao 10 m.

### 1.4. Tính độ lớn tốc độ gió

Tốc độ gió được tính bằng:

$$
w_{t,i,j}=\sqrt{u_{t,i,j}^{2}+v_{t,i,j}^{2}}
$$

Tương ứng với hàm `numpy.hypot()` trong mã nguồn.

Kết quả là trường tốc độ gió vô hướng, không mang thông tin trực tiếp về hướng gió.

### 1.5. Lựa chọn thời gian

Notebook chỉ lấy 400 thời điểm đầu tiên:

$$
W\in\mathbb{R}^{400\times33\times41}
$$

Vì dữ liệu có tần suất một giờ, đoạn được lựa chọn tương ứng với 400 giờ liên tiếp đầu tháng 1/2020.

Việc lựa chọn này được giữ nguyên theo thực nghiệm đã triển khai.

### 1.6. Bảo toàn dữ liệu

Notebook đã tạo và lưu dữ liệu từ lần thực nghiệm trước.

Trong quá trình tái cấu trúc repository, không thực hiện lại bước trích xuất, không thay đổi danh sách địa điểm và không ghi đè dataset đã lưu.


In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
import torch


# 1. Xác định thư mục gốc của project
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "experiments" / "02_climate").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc của project."
    )


# 2. Xác định đường dẫn dữ liệu ERA5
experiment_dir = project_root / "experiments" / "02_climate"
data_dir = project_root / "datasets" / "climate" / "raw"

era5_path = data_dir / "era5_germany_jan2020_10m_wind.nc"

if not era5_path.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy tệp ERA5: {era5_path}"
    )


# 3. Đọc dữ liệu ERA5 từ NetCDF
ds = xr.open_dataset(era5_path)

u10 = ds["u10"]
v10 = ds["v10"]


# 4. Xác định tên chiều thời gian
time_dim = next(
    dim for dim in u10.dims
    if dim in ("time", "valid_time")
)


# 5. Tính tốc độ gió từ 400 thời điểm đầu
wind_speed = np.hypot(
    u10.isel({time_dim: slice(0, 400)}),
    v10.isel({time_dim: slice(0, 400)})
)


# 6. Kiểm tra cấu trúc dữ liệu
print("=== ERA5 WIND SPEED ===")

print("File:", era5_path.name)
print("Variables:", list(ds.data_vars))

print("U10 shape:", u10.shape)
print("V10 shape:", v10.shape)

print("Wind speed shape:", wind_speed.shape)
print("Dimensions:", wind_speed.dims)
print("Units:", u10.attrs.get("units", "unknown"))

print("Selected time steps:", wind_speed.sizes[time_dim])


# 7. Phân biệt thời gian nguồn và thời gian được sử dụng
print("\n=== SOURCE TIME RANGE ===")
print("Start:", str(ds[time_dim].values[0]))
print("End:", str(ds[time_dim].values[-1]))

print("\n=== SELECTED TIME RANGE ===")
print("Start:", str(wind_speed[time_dim].values[0]))
print("End:", str(wind_speed[time_dim].values[-1]))

=== ERA5 WIND SPEED ===
File: era5_germany_jan2020_10m_wind.nc
Variables: ['u10', 'v10']
U10 shape: (744, 33, 41)
V10 shape: (744, 33, 41)
Wind speed shape: (400, 33, 41)
Dimensions: ('valid_time', 'latitude', 'longitude')
Units: m s**-1
Selected time steps: 400

=== SOURCE TIME RANGE ===
Start: 2020-01-01T00:00:00.000000000
End: 2020-01-31T23:00:00.000000000

=== SELECTED TIME RANGE ===
Start: 2020-01-01T00:00:00.000000000
End: 2020-01-17T15:00:00.000000000



## 2. Trích xuất tốc độ gió tại 10 địa điểm ở Đức

### 2.1. Mục tiêu

Từ trường tốc độ gió ERA5, notebook trích xuất các chuỗi thời gian tại 10 địa điểm thuộc nước Đức.

Khác với thí nghiệm CMIP6 Wind sử dụng 5 vị trí trên lưới mục tiêu, thí nghiệm ERA5 sử dụng tọa độ địa lý của 10 thành phố.

Các địa điểm được chọn:

1. Berlin.
2. Hamburg.
3. Munich.
4. Cologne.
5. Frankfurt.
6. Stuttgart.
7. Leipzig.
8. Dresden.
9. Hannover.
10. Nuremberg.

Danh sách này được giữ cố định để bảo đảm số lượng và thứ tự kênh đầu vào khi huấn luyện.

### 2.2. Tọa độ mục tiêu

| Địa điểm | Vĩ độ | Kinh độ |
|---|---:|---:|
| Berlin | 52.5200 | 13.4050 |
| Hamburg | 53.5511 | 9.9937 |
| Munich | 48.1351 | 11.5820 |
| Cologne | 50.9375 | 6.9603 |
| Frankfurt | 50.1109 | 8.6821 |
| Stuttgart | 48.7758 | 9.1829 |
| Leipzig | 51.3397 | 12.3731 |
| Dresden | 51.0504 | 13.7373 |
| Hannover | 52.3759 | 9.7320 |
| Nuremberg | 49.4521 | 11.0767 |

Đây là tọa độ mục tiêu dùng để tìm điểm trên lưới ERA5.

Do độ phân giải lưới hữu hạn, tọa độ thực tế được chọn có thể khác tọa độ mục tiêu.

### 2.3. Phương pháp ánh xạ không gian

Notebook sử dụng phép chọn tọa độ gần nhất của `xarray`:

`wind_speed.sel(..., method="nearest")`

Hai mảng tọa độ mục tiêu được khai báo với cùng chiều `site`.

Điều này cho phép lựa chọn theo từng cặp vĩ độ – kinh độ, thay vì tạo tất cả tổ hợp giữa các vĩ độ và kinh độ.

Mỗi địa điểm được ánh xạ tới một ô lưới ERA5 gần nhất theo cơ chế lựa chọn tọa độ của `xarray`.

### 2.4. Các ô lưới đã được lựa chọn

Theo output thực nghiệm đã lưu:

| Địa điểm | Vĩ độ lưới | Kinh độ lưới |
|---|---:|---:|
| Berlin | 52.50 | 13.50 |
| Hamburg | 53.50 | 10.00 |
| Munich | 48.25 | 11.50 |
| Cologne | 51.00 | 7.00 |
| Frankfurt | 50.00 | 8.75 |
| Stuttgart | 48.75 | 9.25 |
| Leipzig | 51.25 | 12.25 |
| Dresden | 51.00 | 13.75 |
| Hannover | 52.50 | 9.75 |
| Nuremberg | 49.50 | 11.00 |

Notebook kiểm tra để bảo đảm 10 địa điểm không ánh xạ trùng cùng một ô lưới.

### 2.5. Cấu trúc dữ liệu sau trích xuất

Sau khi chọn tọa độ, tensor được đưa về dạng:

$$
W_{\mathrm{raw}}\in\mathbb{R}^{400\times10}
$$

Trong đó:

- Mỗi hàng là một thời điểm.
- Mỗi cột là một địa điểm.
- Mỗi phần tử là độ lớn tốc độ gió tại độ cao 10 m.
- Đơn vị là m/s.

Ma trận được chuyển sang kiểu `float32`.

### 2.6. Kiểm tra dữ liệu

Các phép kiểm tra bao gồm:

- Kích thước phải bằng `(400, 10)`.
- Tất cả giá trị phải hữu hạn.
- Mười tọa độ lưới được chọn phải khác nhau.

Kết quả đã ghi nhận:

- Giá trị nhỏ nhất: 0.042 m/s.
- Giá trị lớn nhất: 8.241 m/s.

Không phát hiện NaN hoặc Inf trong ma trận đã trích xuất.

### 2.7. Lưu dữ liệu

Tệp đầu ra:

`data/era5_wind_10sites.pt`

Tệp lưu ma trận tốc độ gió cùng tên địa điểm, tọa độ mục tiêu, tọa độ thực tế, đơn vị và tên tệp nguồn.

Việc lưu metadata không gian giúp xác định ý nghĩa của từng kênh khi phân tích kết quả dự báo.


In [ ]:
# 1. Danh sách 10 địa điểm tại Đức
locations = {
    "Berlin":    (52.5200, 13.4050),
    "Hamburg":   (53.5511, 9.9937),
    "Munich":    (48.1351, 11.5820),
    "Cologne":   (50.9375, 6.9603),
    "Frankfurt": (50.1109, 8.6821),
    "Stuttgart": (48.7758, 9.1829),
    "Leipzig":   (51.3397, 12.3731),
    "Dresden":   (51.0504, 13.7373),
    "Hannover":  (52.3759, 9.7320),
    "Nuremberg": (49.4521, 11.0767),
}

names = list(locations.keys())


# 2. Tạo các mảng tọa độ mục tiêu theo chiều site
target_lats = xr.DataArray(
    [locations[name][0] for name in names],
    dims="site"
)

target_lons = xr.DataArray(
    [locations[name][1] for name in names],
    dims="site"
)


# 3. Chọn ô lưới ERA5 gần nhất với từng địa điểm
selected = wind_speed.sel(
    latitude=target_lats,
    longitude=target_lons,
    method="nearest"
).transpose(time_dim, "site")


# 4. Chuyển dữ liệu sang ma trận NumPy
wind_raw = selected.values.astype(np.float32)

actual_lats = selected["latitude"].values.tolist()
actual_lons = selected["longitude"].values.tolist()


# 5. Kiểm tra dữ liệu đã trích xuất
assert wind_raw.shape == (400, 10)
assert np.isfinite(wind_raw).all()

grid_points = list(zip(actual_lats, actual_lons))

assert len(set(grid_points)) == 10, (
    "Duplicate ERA5 grid points"
)


# 6. Xác định đường dẫn lưu dataset
output_dir = experiment_dir / "data"
output_dir.mkdir(parents=True, exist_ok=True)

save_path = output_dir / "era5_wind_10sites.pt"


# 7. Lưu dữ liệu và metadata vị trí
torch.save({
    "wind_raw": torch.from_numpy(wind_raw),
    "location_names": names,
    "target_latitudes": target_lats.values.tolist(),
    "target_longitudes": target_lons.values.tolist(),
    "grid_latitudes": actual_lats,
    "grid_longitudes": actual_lons,
    "units": "m s**-1",
    "source": era5_path.name,
}, save_path)


# 8. Tổng hợp kết quả trích xuất
print("=== ERA5 WIND SPEED - 10 LOCATIONS ===")
print("Dataset shape:", wind_raw.shape)

print(
    "Value range (m/s):",
    round(float(wind_raw.min()), 3),
    "-",
    round(float(wind_raw.max()), 3)
)

print("\nSelected ERA5 grid points:")

for i, name in enumerate(names):
    print(
        f"{i + 1:2d}. {name:<12} "
        f"({actual_lats[i]:.2f}, {actual_lons[i]:.2f})"
    )

print("\nSaved:", save_path)

=== ERA5 WIND SPEED - 10 LOCATIONS ===
Dataset shape: (400, 10)
Value range (m/s): 0.042 - 8.241

Selected ERA5 grid points:
 1. Berlin       (52.50, 13.50)
 2. Hamburg      (53.50, 10.00)
 3. Munich       (48.25, 11.50)
 4. Cologne      (51.00, 7.00)
 5. Frankfurt    (50.00, 8.75)
 6. Stuttgart    (48.75, 9.25)
 7. Leipzig      (51.25, 12.25)
 8. Dresden      (51.00, 13.75)
 9. Hannover     (52.50, 9.75)
10. Nuremberg    (49.50, 11.00)

Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\era5_wind_10sites.pt



## 3. Tổng kết dữ liệu ERA5 Wind Speed

### 3.1. Kết quả trích xuất

Notebook đã chuẩn bị dữ liệu tốc độ gió gần bề mặt tại 10 địa điểm thuộc nước Đức từ bộ dữ liệu ERA5.

Tệp nguồn chứa 744 quan sát theo giờ trong tháng 1/2020.

Thực nghiệm chỉ sử dụng 400 quan sát đầu tiên.

Ma trận dữ liệu thu được:

$$
W_{\mathrm{raw}}\in\mathbb{R}^{400\times10}
$$

Thông tin dữ liệu:

| Thuộc tính | Giá trị |
|---|---|
| Nguồn | ERA5 |
| Đại lượng | Tốc độ gió tại độ cao 10 m |
| Số thời điểm sử dụng | 400 |
| Số địa điểm | 10 |
| Kiểu dữ liệu | float32 |
| Đơn vị | m/s |
| Giá trị nhỏ nhất | 0.042 |
| Giá trị lớn nhất | 8.241 |

Dữ liệu vượt qua các phép kiểm tra về kích thước, giá trị hữu hạn và tính duy nhất của các ô lưới đã lựa chọn.

### 3.2. Nội dung tệp đầu ra

Tệp:

`data/era5_wind_10sites.pt`

Chứa các trường:

| Trường | Nội dung |
|---|---|
| `wind_raw` | Tensor tốc độ gió `(400, 10)` |
| `location_names` | Tên 10 địa điểm |
| `target_latitudes` | Vĩ độ mục tiêu |
| `target_longitudes` | Kinh độ mục tiêu |
| `grid_latitudes` | Vĩ độ ô lưới ERA5 đã chọn |
| `grid_longitudes` | Kinh độ ô lưới ERA5 đã chọn |
| `units` | Đơn vị tốc độ gió |
| `source` | Tên tệp NetCDF nguồn |

### 3.3. Liên hệ với notebook Training

Dữ liệu đầu ra được sử dụng trong:

`07_climate_era5_training.ipynb`

Notebook tiếp theo phụ trách:

1. Đọc dữ liệu tốc độ gió đã trích xuất.
2. Chuẩn hóa và tạo các cửa sổ trượt.
3. Nạp kiến trúc mô hình từ notebook Climate Model.
4. Huấn luyện LSTM, PatchTST, Autoformer và Informer.
5. Lưu checkpoint và các metric.
6. Trực quan hóa dự báo và quá trình huấn luyện.

### 3.4. Khác biệt so với CMIP6 Wind

Thí nghiệm CMIP6 Wind trước đó sử dụng dữ liệu mô phỏng khí hậu tại 5 vị trí, với tần suất ba giờ.

Thí nghiệm ERA5 sử dụng dữ liệu tái phân tích khí tượng tại 10 địa điểm, với tần suất một giờ.

Vì vậy, hai thí nghiệm khác nhau về:

- Nguồn dữ liệu.
- Độ phân giải thời gian.
- Số kênh.
- Cách lựa chọn vị trí không gian.

Các kết quả cần được đánh giá trong đúng bối cảnh dữ liệu tương ứng.

### 3.5. Bảo toàn kết quả

Tệp `era5_wind_10sites.pt` đã được tạo trong lần thực nghiệm trước.

Trong quá trình tái cấu trúc repository:

- Không trích xuất lại dữ liệu.
- Không thay đổi 10 địa điểm.
- Không thay đổi thứ tự các kênh.
- Không thay đổi công thức tính tốc độ gió.
- Không ghi đè tệp `.pt`.
- Không chuyển bước chuẩn hóa vào notebook Data.

Notebook được giữ lại nhằm tài liệu hóa quy trình và bảo đảm khả năng tái lập thực nghiệm.